In [ ]:
import sys
import pandas as pd
import pickle
import importlib
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.utils import get_properties

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')

# Add path to the other project src to handle imports with dashes in folder names
sys.path.insert(0, '../../../../../probabilistic_suffix_prediction_U-ED-LSTM/src')

In [ ]:
# log as csv
event_log_path = '../../../../data/data/helpdesk.csv'
event_log_df = pd.read_csv(event_log_path)

case_id_key='CaseID'
activity_key='Activity'
time_key='CompleteTimestamp'

# split csv data
train_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_train.csv'
helpdesk_train_df = pd.read_csv(train_csv_path)

val_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_val.csv'
helpdesk_val_df = pd.read_csv(val_csv_path)

test_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_test.csv'
helpdesk_test_df = pd.read_csv(test_csv_path)

# get case ids
unique_list_train = helpdesk_train_df[case_id_key].dropna().unique().tolist()
unique_list_val = helpdesk_val_df[case_id_key].dropna().unique().tolist()
case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))
case_ids = sorted(case_ids)
print(case_ids[0:5])

# edit and adjust the dataframe event log
df = event_log_df.copy()
if case_ids is not None:
   df = df[df[case_id_key].isin(set(case_ids))]

rename_map = {}
if case_id_key in df.columns and case_id_key != "case:concept:name":
   rename_map[case_id_key] = "case:concept:name"
if activity_key in df.columns and activity_key != "concept:name":
   rename_map[activity_key] = "concept:name"
if time_key in df.columns and time_key != "time:timestamp":
   rename_map[time_key] = "time:timestamp"

if rename_map:
   df = df.rename(columns=rename_map)

# Ensure timestamp is datetime
if "time:timestamp" in df.columns:
   df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], errors="coerce")

# Sort for determinism (important for trace attribute replication + alignments)
sort_cols = ["case:concept:name"]
if "time:timestamp" in df.columns:
   sort_cols.append("time:timestamp")
df = df.sort_values(sort_cols).reset_index(drop=True)

print(df.head(5))

In [ ]:
# Minimal sanity check: timestamps must be datetime + sorted within each case for event_elapsed_time
case_col = "case:concept:name"
ts_col = "time:timestamp"

if ts_col in df.columns:
    is_dt = pd.api.types.is_datetime64_any_dtype(df[ts_col])
    print("time:timestamp is datetime:", is_dt)
    if not is_dt:
        print("WARNING: time:timestamp is not datetime; event_elapsed_time will be wrong")

if case_col in df.columns and ts_col in df.columns:
    neg = df.groupby(case_col)[ts_col].diff() < pd.Timedelta(0)
    n_bad = int(neg.fillna(False).sum())
    print("Out-of-order timestamps (within case):", n_bad)
    if n_bad > 0:
        print("WARNING: df is not sorted within cases; sort upstream before mining")

In [ ]:
# import petri net 
petri_net_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/petri_net/helpdesk.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)

In [ ]:
# Input: GenerateTransitionGuards Algorithm

# - N = (P,T,F): A Petri net without data
petri_net=(net, im, fm)

# petri_net = (net, im, fm)
net, im, fm = petri_net

# - EL: event log
event_log_df = df

# - A: A multiset of optimal control-flow alignments of N and an event log
params = get_properties(event_log_df)  # keeps case_id/activity/timestamp keys, etc.
params["ret_tuple_as_trans_desc"] = True  # <--- key part

aligned_traces = alignments.apply(event_log_df, net, im, fm, parameters=params)
# get alignments:
alignments = [align_dict['alignment'] for align_dict in aligned_traces]
print(alignments[0:5])

# attributes to be considered for decision mining:
dynamic_attributes = ["Resource",
                      # "case_elapsed_time",
                      "event_elapsed_time"]
static_attributes =  ["VariantIndex",
                      "seriousness",
                      # "customer",
                      # "product",
                      # "responsible_section",
                      # "seriousness_2",
                      # "service_level",
                      # "service_type",
                      # "support_section",
                      # "workgroup"
                      ]

attributes = dynamic_attributes + static_attributes

In [ ]:
import decision_mining.custom_framework.decision_discovery
importlib.reload(decision_mining.custom_framework.decision_discovery)
from decision_mining.custom_framework.decision_discovery import DecisionDiscovery, ModelConfig

dd = DecisionDiscovery(petri_net=(net, im, fm),
                       sorted_case_ids = case_ids,
                       event_log_df=event_log_df,
                       alignments=alignments)

# Majority downsampling + aggressive trimming of empty majority rows
mcnfg = ModelConfig(
    model_type="sklearn_tree",
    use_oss=True,
    oss_max_ratio=1.0,
    oss_majority_empty_frac=0.02,
    oss_majority_label_regex=r"^(skip_|tau|silent)",
    use_inverse_freq_weights=True,
    max_depth=4,
    min_samples_leaf=20,
    ccp_alpha=0.0001,
    calibrate=False,
    random_state=7,
 )

res = dd.mine_decision_models(dynamic_attributes=dynamic_attributes, static_attributes=static_attributes, model_cfg=mcnfg)

guards = dd.extract_probabilistic_guards(
    mining_result=res,
    min_leaf_prob=0.20,
    min_leaf_lift=2.0,
    min_leaf_support=20,
    surrogate_max_depth=4,
    surrogate_min_leaf=20,
    always_keep_best=True)

In [ ]:
# Minimal sanity check: ensure VariantIndex is one-hot (categorical) in at least one trained model
if res.models:
    first_model = next(iter(res.models.values())).estimator
    fn = getattr(first_model, "feature_names", [])
    print("Encoded feature count (example place):", len(fn))
else:
    print("No models trained (res.models is empty)")

In [ ]:
"""
Example:
p=0.667; n=90; lift=27.14; rule=(Resource=nan <= 0.5)

Interpretation at this place p_X (decision point):
    - rule: the condition on prefix features that defines a leaf/region of the decision tree.
    - p: estimated probability that the target transition (the header above the guard) is chosen given the rule holds.
    - n: number of training samples (prefixes) that satisfied the rule (support).
    - lift: how much more likely that transition is under the rule compared to its baseline probability at that decision point.
"""

import re

def _format_guard(g):
    parts = []
    if g.get("intervals"):
        parts.append("intervals=" + str(g["intervals"]))
    if g.get("categorical_sets"):
        parts.append("cats=" + str(g["categorical_sets"]))
    parts.append(f"p={g.get('prob', 0.0):.3f}")
    if "support" in g:
        parts.append(f"n={g.get('support', 0)}")
    if "lift" in g:
        parts.append(f"lift={g.get('lift', 0.0):.2f}")
    rule = g.get("rule", "")
    if rule and rule != "(true)":
        parts.append("rule=" + rule)
    return "; ".join(parts)

# Optional: exclude likely-silent transitions from printing.
# Adjust this regex to match your silent transition naming convention.
silent_label_rx = re.compile(r"^(skip_|tau|silent)")

for place_name, by_label in guards.items():
    print(f"\n=== {place_name} ===")
    for label, guard_list in by_label.items():
        if silent_label_rx.search(str(label)):
            continue
        print(f"- {label} ({len(guard_list)} guards)")
        for g in guard_list:
            print("  *", _format_guard(g))

In [ ]:
# Visualize the Petri net with guard annotations
from pm4py.visualization.petri_net import visualizer as pn_vis

def _best_guard_text(guard_list):
    if not guard_list:
        return ""
    # pick guard with highest probability (fallback to first)
    best = max(guard_list, key=lambda g: float(g.get("prob", 0.0)))
    rule = best.get("rule", "")
    prob = best.get("prob", 0.0)
    if rule and rule != "(true)":
        return f"{rule} | p={prob:.2f}"
    return f"p={prob:.2f}"

# Build a transition -> guard text map
transition_guard_text = {}
for place_name, by_label in guards.items():
    for label, guard_list in by_label.items():
        transition_guard_text[str(label)] = _best_guard_text(guard_list)

decorations = {}
for t in net.transitions:
    name = str(t.name)
    guard_text = transition_guard_text.get(name, "")
    label = f"{name}\n{guard_text}" if guard_text else name
    decorations[t] = {"label": label}

params = {"decorations": decorations}
gviz = pn_vis.apply(net, im, fm, parameters=params)
pn_vis.view(gviz)